# Barranquilla MOPAC: el límite de pila de CUDA y el QM/MM con regiones QM pequeñas

En `qmmm_gpu_floor_a100` (A100), con la región QM de 10 átomos, el kernel del gradiente de pares en la GPU tardó
~95 ms por paso, casi todo el piso de ~0,13 s/paso. En `qmmm_gpu_stack_a100` (que corrió en una **L4**), devolver el
límite de pila de CUDA después de cada llamada costaba ~10 ms por gradiente; no devolverlo, o fijarlo alto desde
Python, lo quitaba.

Desde el commit `52c7f964`, la sesión QM/MM usa `MOPAC_GPU_STACK_RESTORE=d` por defecto, y
`MOPAC_GPU_STACK_RESTORE=all` vuelve al comportamiento anterior. Este notebook repite la prueba **en una A100**:

| Bloque | Qué mide | Tiempo (A100) |
|---|---|---|
| A. 10 átomos QM | `gpu_old` (`MOPAC_GPU_STACK_RESTORE=all`), `gpu_new` (por defecto), `gpu_preset` (como `gpu_old`, pero con el límite de pila fijado alto desde Python al inicio) y `cpu`; límite de pila y memoria libre de la GPU después de cada llamada a MOPAC | ~6 min |
| B. Barrido | chignolina con los residuos 1..k (k = 2, 3, 4, 6, 8, 10): `gpu_new` y `cpu` (k ≤ 6) | ~12 min |
| C. Tripsina–BPTI | `gpu_old` y `gpu_new`, 400 pasos de 0,5 fs | ~6 min |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*, y luego *Ejecutar todo*. Si Colab
asigna otra GPU, la celda 2 se detiene. Los
resultados se guardan en Google Drive (`MyDrive/mopac_paper_bench/RESULTS_NAME`); la última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = '52c7f964'            # QM/MM: MOPAC_GPU_STACK_RESTORE=d por defecto; =all, como antes
RESULTS_NAME = 'qmmm_gpu_stack2_a100'
USE_DRIVE = True
SEED = 11
DIAG_STEPS = 300
STACK_PRESET = 65536           # bytes por hilo, variante gpu_preset (A100: 65536 x 221184 hilos ~ 14 GB)
QUICK = False                  # True: 20 pasos (prueba del notebook)
REQUIRE_GPU = 'A100'           # None: aceptar cualquier GPU

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json, gzip
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}. Cambie el tipo de entorno a {REQUIRE_GPU} '
                     f'(o ponga REQUIRE_GPU = None) y vuelva a ejecutar todo.')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/mopac_paper_bench') / RESULTS_NAME
else:
    OUT = Path('/content/mopac_paper_bench') / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC (biblioteca) con OpenBLAS

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
if not head.startswith(COMMIT):
    raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
stamp = OUT / 'commit.txt'
if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
    raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
stamp.write_text(head + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
print(f'compilado en {time.time() - t0:.0f} s: {LIBMOPAC}')

## 4. OpenMM y CuPy

In [ ]:
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} 2>&1 | tail -2", check=False)
run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
    f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available; "
    f"print('QM-PME grid on the GPU (CuPy):', gpu_available())\"", check=False)

## 5. Lanzador con tiempos por llamada

`/content/qmmm_prof.py` ejecuta `run_qmmm_md.py` sin cambiarlo y envuelve cuatro métodos para medir cada llamada:
`MopacQMMMForce.compute` (toda la fuerza QM/MM), `QMSession.step` (MOPAC), `_pme_far_potential` y
`_pme_far_gradient` (PME). El tiempo entre el final de una llamada a la fuerza y el comienzo de la siguiente es el
resto del paso (OpenMM). Los tiempos se guardan en `pytimes.json`.

Modos: `cpu` = `MOPAC_NOGPU=1`; `gpu` = `MOPAC_FORCEGPU=1`. Cada variante añade variables de entorno o opciones.
Con `MOPAC_GPU_PROFILE=1`, MOPAC escribe en su salida (`MOPAC_API_OUTPUT`) el tiempo acumulado por sección después
de cada paso; se guarda el último valor de cada sección en `sections.csv` y la salida completa comprimida.

In [ ]:
PROF = Path('/content/qmmm_prof.py')
PROF.write_text('''
import json, os, runpy, sys, time
script = sys.argv[1]
sys.argv = sys.argv[1:]
sys.path.insert(0, os.path.dirname(os.path.abspath(script)))
import openmm_mopac as om
import mopac_qmmm_api as api
T = {"t0": [], "compute": [], "mopac": [], "pme_far": [], "pme_grad": [], "stack_before": [], "stack_after": [],
     "free_mb_after": [], "preset": None}
rt = None
if os.environ.get("QMMM_STACK_LOG") or os.environ.get("QMMM_STACK_PRESET"):
    try:
        import cupy
        rt = cupy.cuda.runtime
        rt.free(rt.malloc(8))                  # create the primary context (the one MOPAC uses)
        if os.environ.get("QMMM_STACK_PRESET"):
            rt.deviceSetLimit(0, int(os.environ["QMMM_STACK_PRESET"]))   # 0: cudaLimitStackSize
            T["preset"] = rt.deviceGetLimit(0)
        print("stack limit at start:", rt.deviceGetLimit(0), flush=True)
    except Exception as e:
        print("stack log unavailable:", e, flush=True)
        rt = None
def stack_log(key):
    if rt is not None:
        T[key].append(rt.deviceGetLimit(0))
        if key == "stack_after":
            T["free_mb_after"].append(rt.memGetInfo()[0] / 2**20)
def wrap(cls, name, key, stamp=False):
    f = getattr(cls, name)
    def g(self, *a, **k):
        t = time.perf_counter()
        if stamp:
            T["t0"].append(t)
        try:
            return f(self, *a, **k)
        finally:
            T[key].append(time.perf_counter() - t)
    setattr(cls, name, g)
wrap(om.MopacQMMMForce, "compute", "compute", True)
wrap(om.MopacQMMMForce, "_pme_far_potential", "pme_far")
wrap(om.MopacQMMMForce, "_pme_far_gradient", "pme_grad")
wrap(api.QMSession, "step", "mopac")
_step = api.QMSession.step
def _logged(self, *a, **k):
    stack_log("stack_before")
    try:
        return _step(self, *a, **k)
    finally:
        stack_log("stack_after")
api.QMSession.step = _logged
try:
    runpy.run_path(script, run_name="__main__")
finally:
    out = os.environ.get("QMMM_PROF_OUT")
    if out:
        with open(out, "w") as fh:
            json.dump(T, fh)
''')

RX = {
    'system': re.compile(r'system: (\d+) atoms, (\d+) residues, box ([0-9.]+) A'),
    'qm': re.compile(r'QM region: (\d+) atoms \+ (\d+) link atoms, charge (-?\d+); long-range QM-MM electrostatics: (.+)'),
    'nvt': re.compile(r'QM/MM NVT: (\d+) steps x ([0-9.]+) fs in ([0-9.]+) s = ([0-9.]+) s/step, ([0-9.]+) ps/day'),
    'nve': re.compile(r'NVE check (\d+) steps: drift (-?[0-9.]+) kJ/mol, slope of a linear fit (-?[0-9.]+) kJ/mol/ps, '
                      r'largest step ([0-9.]+), rms step ([0-9.]+), rms deviation from the fit ([0-9.]+)'),
    'sec': re.compile(r'\[PROFILE\] MOZYME_SECTION name=(\S+) calls=(\d+) ms=([0-9.]+)'),
}
QMMM = SRC / 'scripts' / 'qmmm' / 'run_qmmm_md.py'

def py_breakdown(T, first, last):
    # medians (ms) of the per-call times over calls first..last-1 (the NVT steps)
    import numpy as np
    c_all, t0 = np.array(T['compute']), np.array(T['t0'])
    c = c_all[first:last]
    # rest of the step: from the end of one force call to the start of the next
    j = np.arange(first, min(last, len(t0) - 1))
    gap = t0[j + 1] - (t0[j] + c_all[j])
    r = {'py_compute_ms': np.median(c) * 1e3 if len(c) else None,
         'py_compute_mean_ms': np.mean(c) * 1e3 if len(c) else None,
         'py_mopac_ms': np.median(T['mopac'][first:last]) * 1e3 if T['mopac'][first:last] else None,
         'py_mopac_mean_ms': np.mean(T['mopac'][first:last]) * 1e3 if T['mopac'][first:last] else None,
         'py_pme_ms': (np.median(np.array(T['pme_far'][first:last]) + np.array(T['pme_grad'][first:last])) * 1e3
                       if T['pme_far'][first:last] else 0.0),
         'py_rest_ms': np.median(gap) * 1e3 if len(gap) else None}
    return {k: (round(float(v), 2) if v is not None else None) for k, v in r.items()}

def qmmm(exp, case, variant, pdb, opts, steps, nve=0, mm_ps=10.0, env_extra=None):
    mode = variant.split('_')[0]
    env_extra = dict(env_extra or {})
    d = OUT / 'runs' / exp / case / variant
    cmd = (f'{sys.executable} -u {PROF} {QMMM} {pdb} --libmopac {LIBMOPAC} {opts} --platform CUDA '
           f'--steps {steps} --nve {nve} --mm-equil-ps {mm_ps} --seed {SEED}')
    key = cmd + '\n' + json.dumps(env_extra, sort_keys=True)
    if (d / 'result.json').exists() and (d / 'cmd.txt').exists() and (d / 'cmd.txt').read_text() == key:
        r = json.loads((d / 'result.json').read_text())
        print(f'[{exp}/{case}] {variant}: ya hecho, {r.get("s_per_step")} s/step')
        return r
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    (d / 'cmd.txt').write_text(key)
    api_out = Path('/tmp') / f'mopac_api_{exp}_{case}_{variant}.out'
    api_out.unlink(missing_ok=True)
    env = {k: v for k, v in os.environ.items() if k not in ('MOPAC_NOGPU', 'MOPAC_FORCEGPU')}
    env.update(OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_GPU_DEBUG='1',
               MOPAC_API_OUTPUT=str(api_out), QMMM_PROF_OUT=str(d / 'pytimes.json'))
    env['MOPAC_NOGPU' if mode == 'cpu' else 'MOPAC_FORCEGPU'] = '1'
    env.update(env_extra)
    env = {k: v for k, v in env.items() if v is not None}       # None: remove the variable
    print(f'[{exp}/{case}] {variant} {env_extra}', flush=True)
    t0 = time.perf_counter()
    with open(d / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, cwd=d, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    text = (d / 'log.txt').read_text(errors='ignore')
    r = {'exp': exp, 'case': case, 'variant': variant, 'mode': mode, 'env': json.dumps(env_extra), 'rc': rc,
         'total_s': round(time.perf_counter() - t0, 1)}
    if (m := RX['system'].search(text)):
        r.update(total_atoms=int(m.group(1)), box_A=float(m.group(3)))
    if (m := RX['qm'].search(text)):
        r.update(qm_atoms=int(m.group(1)), link_atoms=int(m.group(2)), qm_charge=int(m.group(3)), long_range=m.group(4).strip())
    if (m := RX['nvt'].search(text)):
        r.update(steps=int(m.group(1)), dt_fs=float(m.group(2)), md_s=float(m.group(3)),
                 s_per_step=float(m.group(4)), ps_per_day=float(m.group(5)))
    if (m := RX['nve'].search(text)):
        r.update(nve_steps=int(m.group(1)), nve_drift_kj=float(m.group(2)), nve_slope_kj_ps=float(m.group(3)),
                 nve_rms_dev_kj=float(m.group(6)))
    if (d / 'pytimes.json').exists():
        T = json.loads((d / 'pytimes.json').read_text())
        r['force_calls'] = len(T['compute'])
        r.update(py_breakdown(T, 2, 2 + steps))          # the first 2 calls: first SCF from the Lewis structure
        if T.get('stack_after'):
            sb, sa, fm = T['stack_before'], T['stack_after'], T['free_mb_after']
            r.update(stack_preset=T.get('preset'), stack_first_before=sb[0], stack_first_after=sa[0],
                     stack_last_after=sa[-1], stack_max=max(sa + sb), free_mb_first=round(fm[0]),
                     free_mb_last=round(fm[-1]), free_mb_min=round(min(fm)))
    lines = api_out.read_text(errors='ignore').splitlines() if api_out.exists() else []
    r['mozyme_gpu_lines'] = sum('[MOZYME GPU' in l for l in lines)
    r['gpu_scf_stdout_lines'] = sum('[MOZYME GPU SCF]' in l for l in text.splitlines())
    sec = {}
    for l in lines:
        if (m := RX['sec'].search(l)):
            sec[m.group(1)] = (int(m.group(2)), float(m.group(3)))
    if sec:
        with open(d / 'sections.csv', 'w') as fh:
            fh.write('section,calls,ms_total\n')
            for k, (c, ms) in sorted(sec.items(), key=lambda x: -x[1][1]):
                fh.write(f'{k},{c},{ms:.3f}\n')
    if api_out.exists():
        with open(api_out, 'rb') as fi, gzip.open(d / 'mopac_api.out.gz', 'wb') as fo:
            shutil.copyfileobj(fi, fo)
        api_out.unlink()
    (d / ('result.json' if rc == 0 and 's_per_step' in r else 'result_failed.json')).write_text(json.dumps(r, indent=1))
    print(f'    {r.get("s_per_step")} s/step; per call (ms, median): force {r.get("py_compute_ms")}, MOPAC '
          f'{r.get("py_mopac_ms")}, PME {r.get("py_pme_ms")}, rest {r.get("py_rest_ms")}; [MOZYME GPU] lines '
          f'{r["mozyme_gpu_lines"]}, rc {rc}, {r["total_s"]:.0f} s', flush=True)
    if rc != 0:
        print('\n'.join(text.splitlines()[-10:]))
    return r

def get_pdb(code):
    import urllib.request
    f = OUT / 'runs' / '_pdb' / f'{code}.pdb'
    f.parent.mkdir(parents=True, exist_ok=True)
    if not f.exists():
        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{code}.pdb', f)
    return f

CHIG = SRC / 'tests' / 'data' / 'mozyme_gpu_reference' / '1UAO_opt.pdb'
ENZ = ('--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --variants E:57=HID --padding 1.0 --ron 10 --roff 12 '
       '--distances E:195:OG-I:15:C,E:57:NE2-E:195:OG,E:57:HD1-E:102:OD2')
ENZ_PDB = get_pdb('2PTC')
NSTEP = 20 if QUICK else DIAG_STEPS

## A. Región QM de 10 átomos (chignolina, residuo 1)

`QMMM_STACK_LOG=1` hace que el envoltorio lea, con CuPy, el límite de pila y la memoria libre de la GPU antes y después
de cada llamada a MOPAC (CuPy y MOPAC comparten el contexto primario de CUDA). `gpu_preset` fija el límite alto antes
de que MOPAC se cargue, con el comportamiento anterior.

In [ ]:
CHIG = SRC / 'tests' / 'data' / 'mozyme_gpu_reference' / '1UAO_opt.pdb'
NEW = {'MOPAC_GPU_STACK_RESTORE': None}    # por defecto
OLD = {'MOPAC_GPU_STACK_RESTORE': 'all'}
LOG = {'QMMM_STACK_LOG': '1'}
def chig_opts(k):
    return f'--qm-residues {",".join(map(str, range(1, k + 1)))} --padding 2.5 --ron 6 --roff 8'
res = []
for variant, envx in (('gpu_old', {**LOG, **OLD}), ('gpu_new', {**LOG, **NEW}),
                      ('gpu_preset', {**LOG, **OLD, 'QMMM_STACK_PRESET': str(STACK_PRESET)}), ('cpu', {})):
    res.append(qmmm('A_small', 'chig_1-1', variant, CHIG, chig_opts(1), NSTEP, env_extra=envx))

## B. Barrido del tamaño de la región QM

`gpu_new` para k = 2, 3, 4, 6, 8, 10 y `cpu` para k = 2, 3, 4, 6 (la CPU con k ≥ 8 ya se midió: 1,05 y 1,81 s/paso).

In [ ]:
for k in ([2] if QUICK else [2, 3, 4, 6, 8, 10]):
    res.append(qmmm('B_scan', f'chig_1-{k}', 'gpu_new', CHIG, chig_opts(k), NSTEP, env_extra=NEW))
    if k <= 6:
        res.append(qmmm('B_scan', f'chig_1-{k}', 'cpu', CHIG, chig_opts(k), NSTEP))

## C. Tripsina–BPTI

Las opciones de la corrida del artículo, 400 pasos de 0,5 fs, con el comportamiento anterior y con el nuevo.

In [ ]:
ENZ = ('--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --variants E:57=HID --padding 1.0 --ron 10 --roff 12 '
       '--distances E:195:OG-I:15:C,E:57:NE2-E:195:OG,E:57:HD1-E:102:OD2')
ENZ_PDB = get_pdb('2PTC')
for variant, envx in (('gpu_old', {**LOG, **OLD}), ('gpu_new', {**LOG, **NEW})):
    res.append(qmmm('C_trypsin', 'trypsin_pme', variant, ENZ_PDB, ENZ, 20 if QUICK else 400,
                    mm_ps=10.0 if QUICK else 100.0, env_extra=envx))

## 6. Resultados

In [ ]:
import pandas as pd
rows = []
for f in sorted((OUT / 'runs').rglob('result*.json')):
    r = json.loads(f.read_text()); r['ok'] = f.name == 'result.json'; rows.append(r)
df = pd.DataFrame(rows)
df['qm_total'] = df['qm_atoms'] + df['link_atoms']
df.to_csv(OUT / 'qmmm_stack.csv', index=False)
cols = ['exp', 'case', 'variant', 'qm_total', 's_per_step', 'ps_per_day', 'py_mopac_ms', 'py_pme_ms', 'py_compute_ms',
        'py_rest_ms', 'stack_preset', 'stack_first_before', 'stack_first_after', 'stack_last_after', 'stack_max',
        'free_mb_first', 'free_mb_last', 'mozyme_gpu_lines', 'ok']
display(df[[c for c in cols if c in df]].sort_values(['exp', 'qm_total', 'variant']))

## 7. Descargar

In [ ]:
tmp = Path('/content/_qmmm_stack2') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for f in ('qmmm_stack.csv', 'env.json', 'commit.txt'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
for pat in ('*.json', 'log.txt', 'cmd.txt', 'sections.csv', 'nsys_stats.txt', 'mopac_api.out.gz'):
    for f in (OUT / 'runs').rglob(pat):
        dst = tmp / 'runs' / f.relative_to(OUT / 'runs')
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', tmp.parent)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
from google.colab import files
files.download(z)